In [3]:
import os
import hashlib
from collections import defaultdict
from PIL import Image

def validate_dataset(dataset_dir):
    corrupted = []
    grayscale = []
    non_rgb = []
    dimensions = defaultdict(int)
    duplicates = defaultdict(list)
    class_counts = defaultdict(int)
    empty_folders = []

    # Hash set for duplicate detection
    seen_hashes = {}

    for root, dirs, files in os.walk(dataset_dir):
        if not files:
            empty_folders.append(root)
            continue

        class_name = os.path.basename(root)
        class_counts[class_name] += len(files)

        for file in files:
            file_path = os.path.join(root, file)
            try:
                with Image.open(file_path) as img:
                    # Check mode
                    if img.mode != "RGB":
                        non_rgb.append(file_path)
                        if img.mode == "L":
                            grayscale.append(file_path)

                    dimensions[img.size] += 1

                    file_hash = hashlib.md5(img.tobytes()).hexdigest()
                    if file_hash in seen_hashes:
                        duplicates[file_hash].append(file_path)
                    else:
                        seen_hashes[file_hash] = file_path

            except Exception:
                corrupted.append(file_path)

    # Summary
    print("=== Dataset Validation Report ===")
    print(f"Corrupted images: {len(corrupted)}")
    print(f"Non-RGB images: {len(non_rgb)}")
    print(f"Grayscale images: {len(grayscale)}")
    print("Image dimensions distribution:")
    for dim, count in dimensions.items():
        print(f"  {dim}: {count}")
    print("Duplicate images found:")
    for h, paths in duplicates.items():
        print(f"  Hash {h}: {paths}")
    print("Class counts:")
    for cls, count in class_counts.items():
        print(f"  {cls}: {count}")
    print("Empty folders:")
    for folder in empty_folders:
        print(f"  {folder}")

# Example usage
validate_dataset(r"D:\AgriForge\datasets\raw\tomato")


=== Dataset Validation Report ===
Corrupted images: 0
Non-RGB images: 0
Grayscale images: 0
Image dimensions distribution:
  (256, 256): 15095
Duplicate images found:
Class counts:
  bacterial_spot: 2127
  early_blight: 1000
  healthy: 1585
  late_blight: 1901
  leaf_mold: 952
  mosaic_virus: 1343
  septoria_leaf_spot: 1771
  spider_mites: 1676
  target_spot: 1404
  yellow_leaf_curl_virus: 1336
Empty folders:
  D:\AgriForge\datasets\raw\tomato


In [21]:
from PIL import Image
import os

dataset_path = "D:/AgriForge/datasets/raw/tomato"

corrupted_files = []

for root, _, files in os.walk(dataset_path):
    for file in files:
        file_path = os.path.join(root, file)
        try:
            # Try opening the image
            img = Image.open(file_path)
            img.verify()  # Verify integrity
        except Exception as e:
            print(f"Corrupted image detected: {file_path} ({e})")
            corrupted_files.append(file_path)

# Remove corrupted files
for f in corrupted_files:
    os.remove(f)
    print(f"Removed: {f}")


In [22]:
import hashlib, os

def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

seen = {}
for root, _, files in os.walk("D:/AgriForge/datasets/raw/tomato"):
    for file in files:
        path = os.path.join(root, file)
        h = file_hash(path)
        if h in seen:
            print("Duplicate:", path, "==", seen[h])
            os.remove(path)  # or move to a duplicates folder
        else:
            seen[h] = path


Duplicate: D:/AgriForge/datasets/raw/tomato\mosaic_virus\aug_276.jpg == D:/AgriForge/datasets/raw/tomato\mosaic_virus\aug_10.jpg
Duplicate: D:/AgriForge/datasets/raw/tomato\mosaic_virus\aug_300.jpg == D:/AgriForge/datasets/raw/tomato\mosaic_virus\aug_134.jpg
Duplicate: D:/AgriForge/datasets/raw/tomato\mosaic_virus\aug_308.jpg == D:/AgriForge/datasets/raw/tomato\mosaic_virus\aug_254.jpg
Duplicate: D:/AgriForge/datasets/raw/tomato\mosaic_virus\aug_359.jpg == D:/AgriForge/datasets/raw/tomato\mosaic_virus\aug_215.jpg
Duplicate: D:/AgriForge/datasets/raw/tomato\mosaic_virus\aug_457.jpg == D:/AgriForge/datasets/raw/tomato\mosaic_virus\aug_22.jpg
Duplicate: D:/AgriForge/datasets/raw/tomato\mosaic_virus\aug_523.jpg == D:/AgriForge/datasets/raw/tomato\mosaic_virus\aug_25.jpg
Duplicate: D:/AgriForge/datasets/raw/tomato\mosaic_virus\aug_588.jpg == D:/AgriForge/datasets/raw/tomato\mosaic_virus\aug_137.jpg
Duplicate: D:/AgriForge/datasets/raw/tomato\mosaic_virus\aug_615.jpg == D:/AgriForge/datasets

In [16]:
import os
import random
import cv2
import albumentations as A

# =========================================================
# STEP 1: CHANGE THESE PATHS TO MATCH YOUR ACTUAL COMPUTOR
# =========================================================
# Put an 'r' before the quote to handle Windows backslashes properly
source_dir = r"D:\AgriForge\datasets\raw\tomato\mosaic_virus" 
output_dir = r"D:\AgriForge\datasets\raw\tomato\augmented_dataset"

os.makedirs(output_dir, exist_ok=True)

# =========================================================
# STEP 2: DEBUG PRINTS (Let's see what's actually there)
# =========================================================
print(f"Checking directory: {source_dir}")
if not os.path.exists(source_dir):
    print("❌ ERROR: The source directory does not exist! Check your spelling/path.")
else:
    all_files = os.listdir(source_dir)
    print(f"Total files found in folder (any type): {len(all_files)}")
    if len(all_files) > 0:
        print(f"Sample files in folder: {all_files[:5]}")

# Load images using lower case to avoid .JPG vs .jpg issues
images = [f for f in os.listdir(source_dir) if f.lower().endswith(('jpg', 'jpeg', 'png'))]
print(f"Validated image files found: {len(images)}")

# Guard rail to stop execution before the loop crashes
if len(images) == 0:
    raise ValueError("Stopping execution: 'images' list is empty. See debug logs above.")

# =========================================================
# STEP 3: AUGMENTATION PIPELINE
# =========================================================
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=45, p=0.5),
    A.RandomBrightnessContrast(p=0.2),
])

target_count = 1000
generated_count = 0

print("\nStarting augmentation loop...")
while generated_count < target_count:
    img_name = random.choice(images)
    image = cv2.imread(os.path.join(source_dir, img_name))
    
    # Safety check if OpenCV fails to read a specific image file
    if image is None:
        continue
        
    image = cv2.resize(image, (256, 256))
    
    augmented = transform(image=image)
    aug_img = augmented['image']
    
    cv2.imwrite(os.path.join(output_dir, f"aug_{generated_count}.jpg"), aug_img)
    generated_count += 1

print(f"🎉 Success! Generated {generated_count} images in '{output_dir}'")

Checking directory: D:\AgriForge\datasets\raw\tomato\mosaic_virus
Total files found in folder (any type): 373
Sample files in folder: ['000ec6ea-9063-4c33-8abe-d58ca8a88878___PSU_CG 2169.JPG', '006e354b-c054-4b72-a83c-e3feb038942e___PSU_CG 2330.JPG', '00c07a77-15e6-4815-92d4-8d1e1afb7f3c___PSU_CG 2052.JPG', '01b32f27-2b9b-4961-805b-8066bf4d90f1___PSU_CG 2417.JPG', '021accd9-bbb2-4777-8f94-93295e6de49e___PSU_CG 2075.JPG']
Validated image files found: 373

Starting augmentation loop...
🎉 Success! Generated 1000 images in 'D:\AgriForge\datasets\raw\tomato\augmented_dataset'


In [2]:
import os, shutil, random

# Source root where your raw tomato and potato folders live
source_root = "D:/AgriForge/datasets/raw"
crops = ["tomato", "potato"]

# Target root for training pipeline
target_root = "data/crop_disease_dataset"
train_ratio = 0.8  # 80% train, 20% val

for crop in crops:
    crop_path = os.path.join(source_root, crop)
    if not os.path.isdir(crop_path):
        continue
    
    # Loop through each disease class folder
    for cls in os.listdir(crop_path):
        cls_path = os.path.join(crop_path, cls)
        if not os.path.isdir(cls_path):
            continue
        
        images = os.listdir(cls_path)
        random.shuffle(images)
        split = int(len(images) * train_ratio)
        
        train_imgs = images[:split]
        val_imgs = images[split:]
        
        # Create train/val directories
        for subset, subset_imgs in [("train", train_imgs), ("val", val_imgs)]:
            subset_dir = os.path.join(target_root, subset, crop, cls)
            os.makedirs(subset_dir, exist_ok=True)
            for img in subset_imgs:
                src = os.path.join(cls_path, img)
                dst = os.path.join(subset_dir, img)
                shutil.copy(src, dst)

print("✅ Dataset reorganized into train/val structure at:", target_root)


✅ Dataset reorganized into train/val structure at: data/crop_disease_dataset


In [2]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [1]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

2.11.0+cu128
12.8
True
1
